# Basic LLM Parellel Workflow
### -- UPSC Essay Evaluation --

In [64]:
# libraries
from langgraph.graph import StateGraph
from langgraph.graph import START, END
from langchain_groq.chat_models import ChatGroq
from pydantic import BaseModel, Field
from IPython.display import Image
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os
import operator

In [65]:
# load api key
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [66]:
# define LLM model
llm_model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0,
    max_tokens = None
)

In [67]:
# define schema output
class EvaluationSchema(BaseModel):
    feedback: str = Field(description = "Detailed feedback for the essay")
    score: int = Field(description = "Score out of 10", ge = 0, le = 10)

In [68]:
# pass schema to model
structured_model = llm_model.with_structured_output(EvaluationSchema)

In [69]:
essay = """Rise of AI in India

Artificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century, reshaping economies, industries, and societies worldwide. India, with its rapidly growing digital infrastructure, expanding startup ecosystem, and a vast pool of skilled professionals, has positioned itself as a key player in the global AI landscape. The rise of AI in India is not just a technological phenomenon but also a socio-economic revolution with far-reaching implications.

Growth of AI in India

The AI revolution in India has been fueled by several factors. First, the government has actively promoted AI research and adoption through initiatives such as the National AI Strategy by NITI Aayog, which emphasizes AI for inclusive growth. Second, India’s IT sector, already a global leader in software services, has pivoted towards AI-based solutions, creating new opportunities in healthcare, education, agriculture, and finance. Third, the rise of startups focusing on AI-driven products and services has accelerated innovation across sectors.

Applications Across Sectors

AI is making significant inroads into multiple domains in India:

Healthcare: AI-powered diagnostic tools, telemedicine platforms, and predictive analytics are improving patient care, especially in rural areas.

Agriculture: AI-driven crop monitoring, precision farming, and market prediction systems are helping farmers increase yield and reduce losses.

Education: Personalized learning platforms and AI tutors are transforming classrooms into adaptive learning environments.

Finance: AI-based fraud detection, chatbots, and algorithmic trading are revolutionizing the financial sector.

Governance: Smart city projects, facial recognition for security, and AI-driven data analysis are enhancing governance and public services.

Challenges in AI Adoption

Despite its rapid growth, AI in India faces several challenges. A shortage of high-quality annotated datasets, limited awareness among traditional industries, ethical concerns related to bias and privacy, and the need for better AI governance frameworks are pressing issues. Additionally, India must invest heavily in research and development to match the pace of AI innovation globally.

Future Prospects

The future of AI in India looks promising. With a young population, strong IT infrastructure, and increasing government and private investment, India has the potential to become an AI powerhouse. If harnessed responsibly, AI can address critical issues such as healthcare accessibility, agricultural efficiency, and sustainable urbanization, thereby driving inclusive growth.

Conclusion

The rise of AI in India is more than a technological trend; it represents a paradigm shift in how the country approaches development and innovation. By combining AI with its demographic advantage and entrepreneurial spirit, India can not only become a leader in AI adoption but also ensure that the benefits of AI are shared widely across society. The challenge lies in balancing innovation with ethics, ensuring that AI is used for the greater good of all citizens."""

In [70]:
# prompt
prompt = f"Evaluate the language quality of the essay and provide a feedback and assign a score out of 10:\n\n{essay}"
answer = structured_model.invoke(prompt)

In [71]:
# result
print(answer.score)

8


In [72]:
# define state
class EvaluationState(TypedDict):
    essay_text: str
    language_feedback: str
    depth_feedback: str
    clarity_feedback: str
    feedback_scores: Annotated[list[int], operator.add] # merging
    average_score: float 
    summary_feedback: str

In [73]:
# evaluate language function
def evaluate_language(state: EvaluationState):
    prompt = f"Evaluate the language quality of the essay and provide a feedback and assign a score out of 10:\n\n{state["essay_text"]}"
    result = structured_model.invoke(prompt)
    
    return {
        "language_feedback": result.feedback,
        "feedback_scores": [result.score]
    }

In [74]:
# evaluate analysis function
def evaluate_analysis(state: EvaluationState):
    essay = state["essay_text"]

    prompt = f"Evaluate the depth of analysis of the essay and provide a feedback and assign a score out of 10:\n\n{essay}"
    result = structured_model.invoke(prompt)
    
    return {
        "depth_feedback": result.feedback,
        "feedback_scores": [result.score]
    }

In [75]:
# evaluate thought function
def evaluate_thought(state: EvaluationState):
    essay = state["essay_text"]

    prompt = f"Evaluate the clarity of thought of the essay and provide a feedback and assign a score out of 10:\n\n{essay}"
    result = structured_model.invoke(prompt)
    
    return {
        "clarity_feedback": result.feedback,
        "feedback_scores": [result.score]
    }

In [79]:
# final evaluation function
def final_evaluation(state: EvaluationState):
    prompt = f"Based on the following feedbacks create a summarized feedback \n language feedback: {state["language_feedback"]} \n depth of analysis: {state["depth_feedback"]} \n clarity feedback: {state["clarity_feedback"]}"
    
    final_feedback = llm_model.invoke(prompt).content
    average_score = sum(state["feedback_scores"]) / (len(state["feedback_scores"]))

    return {
        "summary_feedback": final_feedback,
        "average_score": average_score
    }

In [80]:
# build graph
graph = StateGraph(EvaluationState)

# nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_feedback", final_evaluation)

# edges
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

graph.add_edge("evaluate_language", "final_feedback")
graph.add_edge("evaluate_analysis", "final_feedback")
graph.add_edge("evaluate_thought", "final_feedback")

graph.add_edge("final_feedback", END)

# compile
workflow = graph.compile()

In [82]:
# execute
initial_state = {
    "essay_text": essay
}
workflow.invoke(initial_state)

{'essay_text': 'Rise of AI in India\n\nArtificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century, reshaping economies, industries, and societies worldwide. India, with its rapidly growing digital infrastructure, expanding startup ecosystem, and a vast pool of skilled professionals, has positioned itself as a key player in the global AI landscape. The rise of AI in India is not just a technological phenomenon but also a socio-economic revolution with far-reaching implications.\n\nGrowth of AI in India\n\nThe AI revolution in India has been fueled by several factors. First, the government has actively promoted AI research and adoption through initiatives such as the National AI Strategy by NITI Aayog, which emphasizes AI for inclusive growth. Second, India’s IT sector, already a global leader in software services, has pivoted towards AI-based solutions, creating new opportunities in healthcare, education, agriculture, and finance. Third, 